* Trabalho final FIS01085 - Ciência de Dados

Gabriel Souto e Pedro Griebler Reche

2025/2

In [ ]:
from ucimlrepo import fetch_ucirepo 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics, datasets
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import PCA
import shap
import seaborn as sns
import statsmodels.formula.api as smf
from imblearn.over_sampling import SMOTE

# fetch dataset 
raisin = fetch_ucirepo(id=850) 
  
# data (as pandas dataframes) 
X = raisin.data.features 
y = raisin.data.targets 
  
# metadata 
print(raisin.metadata) 
  
# variable information 
print(raisin.variables) 

{'uci_id': 850, 'name': 'Raisin', 'repository_url': 'https://archive.ics.uci.edu/dataset/850/raisin', 'data_url': 'https://archive.ics.uci.edu/static/public/850/data.csv', 'abstract': 'Images of the Kecimen and Besni raisin varieties were obtained with CVS. A total of 900 raisins were used, including 450 from both varieties, and 7 morphological features were extracted.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 900, 'num_features': 7, 'feature_types': ['Real', 'Integer'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Fri Jan 05 2024', 'dataset_doi': '10.24432/C5660T', 'creators': ['İ̇lkay Çinar', 'Murat Koklu', 'Sakir Tasdemir'], 'intro_paper': {'ID': 261, 'type': 'NATIVE', 'title': 'Kuru Üzüm Tanelerinin Makine Görüşü ve Yapay Zeka Yöntemleri Kullanılarak Sınıflandırılması', 'authors': 'İ̇lkay Çinar

In [ ]:
def evaluate_tree_model(max_depths, X, y, test_size, seeds):
    accuracies = np.zeros((len(max_depths), len(seeds)))

    # Loop para cada profundidade que você quer testar
    for i, depth in enumerate(max_depths):
        # O modelo é definido uma vez por profundidade com um estado aleatório fixo.
        # Isso garante que a única variável mudando no loop interno é a divisão dos dados.
        model = DecisionTreeClassifier(max_depth=depth, random_state=42) # Usando um valor fixo, como 42

        # Loop para cada semente aleatória para dividir os dados
        for j, seed in enumerate(seeds):
            # Divide os dados usando a semente da vez
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify = y)

            # Treina e avalia o modelo
            model.fit(X_train, y_train)
            accuracies[i, j] = model.score(X_test, y_test) # Armazena a acurácia

    return accuracies

def visualize_tree(X, test_size, seed, feature_names, class_names):

     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify = y)
     clf = DecisionTreeClassifier(max_depth=3, random_state=seed)
     clf.fit(X_train, y_train)

     plt.figure(figsize=(20,10))
     plot_tree(clf, filled=True, feature_names=feature_names, class_names=class_names)
     plt.savefig('decision_tree.png', dpi=500)
     plt.show()


X = df.drop(columns=['mild_cognitive_impairment'])
y = df['mild_cognitive_impairment']

X_scaled = MinMaxScaler().fit_transform(X)

y_scaled = MinMaxScaler().fit_transform(y.values.reshape(-1, 1))

max_depths = [3, 5, 7]
seeds = np.random.randint(0, 1000, size=100)

test_size = 0.2

feature_names = df.drop(columns=['mild_cognitive_impairment']).columns
class_names = ['Negative', 'Positive']

visualize_tree(X, test_size, seeds[0], feature_names, class_names)
accuracies = evaluate_tree_model(max_depths, X, y, test_size, seeds)

plt.figure(figsize=(10, 6))
plt.subplot(221)
plt.hist(accuracies[0, :], edgecolor='black', label='max_depth=3')

plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(222)
plt.hist(accuracies[1, :], label='max_depth=5', edgecolor='black')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(223)
plt.hist(accuracies[2, :], label='max_depth=7', edgecolor='black')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.legend()

plt.show()

In [ ]:
clf = DecisionTreeClassifier ()
grid_parameters = {
'criterion': [ 'gini' , 'entropy'] ,
'max_depth': [3 , 5 , 7 ] ,
'min_samples_leaf': [1 , 5] ,
}
grid_search = GridSearchCV ( estimator = clf , param_grid = grid_parameters , cv
=5)
grid_search.fit (X, y)
print ( " Melhores Parametros : " , grid_search.best_params_ )

In [ ]:
# Normalize all columns except target
target_column = 'mild_cognitive_impairment'

columns_to_normalize = df.columns.difference([target_column])
df[columns_to_normalize] = (df[columns_to_normalize] - df[columns_to_normalize].min()) / (df[columns_to_normalize].max() - df[columns_to_normalize].min())

#Feature distributions splitting diabetes vs no diabetes

plt.figure(figsize=(15, 10))
for i, column in enumerate(columns_to_normalize):
    plt.subplot(5, 4, i + 1)
    #plot as histogram
    sns.histplot(data=df, x=column, hue=target_column, multiple="stack", bins=15, alpha=0.5, stat = 'density', common_norm = False)
    plt.title(f'Distribution of {column}')
plt.tight_layout()
plt.show()

models = {
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=3)
}
df.columns = df.columns.str.strip()

X = df.drop(columns=[target_column])
y = df[target_column]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42, stratify = y)
Accuracy_table = pd.DataFrame()
# Ensure order and types match
X_test = X_test[X_train.columns]
X_train = X_train.astype(float)
X_test = X_test.astype(float)

Accuracy_table = pd.DataFrame()
fig, axes = plt.subplots(2, len(models), figsize=(18, 10))
if len(models) == 1:
    axes = np.array([[axes[0]], [axes[1]]])  # Ensure axes is 2D for consistency

for idx, (model_name, model) in enumerate(models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # SHAP KernelExplainer for all models
    background = shap.sample(X_train, 100, random_state=42)
    explainer = shap.KernelExplainer(model.predict_proba, background)
    shap_values = explainer.shap_values(X_test, nsamples=100)

    # Accuracy metrics table
    metrics_table = pd.DataFrame({
        'Model': model_name,
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        'Score': [
            metrics.accuracy_score(y_test, y_pred),
            metrics.precision_score(y_test, y_pred),
            metrics.recall_score(y_test, y_pred),
            metrics.f1_score(y_test, y_pred)
        ]
    })
    Accuracy_table = pd.concat([Accuracy_table, metrics_table], ignore_index=True)

    # Confusion matrix subplot
    metrics.ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, ax=axes[0, idx], colorbar=True, cmap='Blues'
    )
    axes[0, idx].set_title(f'Confusion Matrix\n{model_name}\nAccuracy: {metrics.accuracy_score(y_test, y_pred):.2f}', pad=15, fontsize=12)
    axes[0, idx].set_xlabel('Predicted label', fontsize=11)
    axes[0, idx].set_ylabel('True label', fontsize=11)

    # SHAP values subplot (feature importance)
    # For new SHAP: shap_values shape is (n_samples, n_features, n_outputs)
    if np.array(shap_values).shape == (X_test.shape[0], X_test.shape[1], 2):
        shap_importance = np.abs(shap_values[:, :, 1]).mean(axis=0)
        shap_df = pd.Series(shap_importance, index=X_test.columns).sort_values(ascending=True)
        shap_df.plot.barh(ax=axes[1, idx], color='teal')
        axes[1, idx].set_title(f'SHAP Feature Importance\n{model_name}', fontsize=12, pad=15)
        axes[1, idx].set_xlabel('Mean(|SHAP value|)', fontsize=11)
        axes[1, idx].set_ylabel('Feature', fontsize=11)
        axes[1, idx].set_xlim(0, 0.1)
    else:
        axes[1, idx].set_visible(False)
        print("Shape mismatch! Check your SHAP values and X_test.")

fig.suptitle("Model Performance and SHAP Feature Importances", fontsize=20, fontweight='bold', y=1.0)
axes[0, 0].set_ylabel('True label', fontsize=12)
axes[1, 0].set_ylabel('Feature', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

print("Accuracy and Metrics Table:")
display(Accuracy_table)

# Pivot the table for plotting
metrics_pivot = Accuracy_table.pivot(index='Metric', columns='Model', values='Score')

plt.figure(figsize=(10, 10))
ax = metrics_pivot.plot(kind='bar', rot=0, width=0.9, colormap='Set2')
plt.title("Model Comparison Across Metrics", fontsize=16, fontweight='bold', pad=15)
plt.ylabel("Score")
plt.ylim(0.5, 1.05)
plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Add value labels on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=8)

plt.show()

In [ ]:
#Criar mapa de correlação